In [124]:
from __future__ import annotations
import json, math, re, unicodedata, uuid, random
from dataclasses import dataclass
from pathlib import Path
from datetime import date, datetime, time, timedelta
import re
import pandas as pd
import numpy as np

In [125]:
PY_SEED = 42
np.random.seed(PY_SEED)
random.seed(PY_SEED)

YEAR = 2025

from datetime import date
FECHA_INI = date(YEAR, 1, 1)
FECHA_FIN = date(YEAR, 7, 31)

N_EMPRESAS_TRANSP = 3
USUARIOS_POR_EMPRESA = 3
TICKETS_POR_USUARIO = 50
PESO_TARJETA = 0.85

DC_THRESHOLD_KW = 20

STATION_OFFSET_MIN, STATION_OFFSET_MAX = -0.02, 0.02
DAILY_NOISE_MIN, DAILY_NOISE_MAX       = -0.01, 0.01

KWH_MIN, KWH_MAX, KWH_MODE = 10.0, 80.0, 35.0

PRODUCTO = "Electricidad"

from pathlib import Path
DATA_DIR = Path(r"data")

PATH_PUNTOS = DATA_DIR / "PuntosCarga.csv"
PATH_OCM    = DATA_DIR / "ocm_agg_2025.csv"
PATH_CIFS   = DATA_DIR / "CIFs_puntos_carga.csv"

COL_STATION_ID      = "station_id"
COL_EMPRESA         = "empresa"
COL_POT_MAX_KW      = "potencia_max_kw"
COL_PRECIO_AC       = "precio_ac_eur_kwh"
COL_PRECIO_DC       = "precio_dc_eur_kwh"
COL_LAT             = "lat"
COL_LON             = "lon"

COL_EMPRESA_CIF_EMP = "empresa"
COL_EMPRESA_CIF_CIF = "cif"

OUT_JSONL = DATA_DIR / "tickets_ev_sinteticos.jsonl"
OUT_JSON  = DATA_DIR / "tickets_ev_sinteticos.json"

In [126]:
def norm_txt(x: str) -> str:
    if pd.isna(x): return ""
    x = str(x).strip()
    x = "".join(c for c in unicodedata.normalize("NFD", x) if unicodedata.category(c) != "Mn")
    return x.lower()

def round2(x: float) -> float:
    return float(np.round(x + 1e-12, 2))

def round3(x: float) -> float:
    return float(np.round(x + 1e-12, 3))

def random_fecha(fecha_ini: date, fecha_fin: date) -> date:
    delta = (fecha_fin - fecha_ini).days
    return fecha_ini + timedelta(days=int(np.random.randint(0, delta + 1)))

def random_hora() -> time:
    return (datetime.min + timedelta(seconds=int(np.random.randint(0, 24*3600)))).time()

def random_kwh() -> float:
    return round2(np.random.triangular(KWH_MIN, KWH_MODE, KWH_MAX))

def elegir_metodo_pago() -> str:
    return "Tarjeta de empresa" if random.random() < PESO_TARJETA else "Efectivo"

def str_fecha(d: date) -> str:
    return d.strftime("%Y-%m-%d")

def str_hora(t: time) -> str:
    return t.strftime("%H:%M:%S")

In [127]:
def _brand_from_text(s: str) -> str:
    s0 = norm_txt(str(s))
    m = [
        ("repsol", "Repsol"),
        ("cepsa", "Cepsa"),
        ("bp", "BP"),
        ("endesa", "Endesa"),
        ("iberdrola", "Iberdrola"),
        ("ionity", "IONITY"),
        ("wenea", "Wenea"),
        ("zunder", "Zunder"),
        ("totalenergies", "TotalEnergies"),
        ("nedgia", "Nedgia"),
    ]
    for needle, brand in m:
        if needle in s0:
            return brand
    return ""

def _brand_from_id(s: str) -> str:
    s1 = str(s).upper()
    if "REP" in s1: return "Repsol"
    if "MOE" in s1 or "CEPSA" in s1 or "MOEV" in s1: return "Cepsa"
    if "IBR" in s1 or "IBERDROLA" in s1: return "Iberdrola"
    if "END" in s1 or "ENDESA" in s1: return "Endesa"
    if "ION" in s1 or "IONITY" in s1: return "IONITY"
    if "WEN" in s1 or "WENEA" in s1: return "Wenea"
    if "ZUN" in s1 or "ZUNDER" in s1: return "Zunder"
    if "BP" in s1: return "BP"
    if "TOT" in s1 or "TOTAL" in s1: return "TotalEnergies"
    return ""

est_std["empresa"] = (
    est_std["empresa"]
    .astype(str)
    .replace({"nan": "", "none": "", "None": ""})
    .str.strip()
)

mask_empty = est_std["empresa"].eq("") | est_std["empresa"].isna()
if mask_empty.any():
    est_std.loc[mask_empty, "empresa"] = est_std.loc[mask_empty, "id_estacion"].map(_brand_from_id)
    mask_empty = est_std["empresa"].eq("") | est_std["empresa"].isna()
    if mask_empty.any() and "direccion" in est_std.columns:
        est_std.loc[mask_empty, "empresa"] = est_std.loc[mask_empty, "direccion"].map(_brand_from_text)

est_std["empresa_norm"] = est_std["empresa"].map(norm_txt)
est_std["empresa"] = est_std["empresa"].replace("", "Operador desconocido")


In [128]:
op_candidates = [c for c in est.columns if re.search(r"(?:^|_)(operador|operator)(?:$|_)", c, flags=re.I)]
if op_candidates:
    pref = [c for c in op_candidates if c in ("operador","operator","operador_nombre","operatorname","operatorinfo","operatorinfotitle")]
    op_col = pref[0] if pref else op_candidates[0]
else:
    op_col = "p1" if "p1" in est.columns else None

if op_col:
    est_std["empresa"] = est[op_col].astype(str).str.strip()
else:
    est_std["empresa"] = est_std.get("empresa", pd.Series([""], index=est_std.index)).astype(str).str.strip()

est_std["empresa"] = est_std["empresa"].fillna("").astype(str).str.strip()
est_std["empresa_norm"] = est_std["empresa"].map(norm_txt)

In [129]:
CONTROL_LETTERS = "JABCDEFGHI"
LETTER_GROUP = set("PQRSNW")
DIGIT_GROUP  = set("ABEH")
ANY_GROUP    = set("CDFGJUVXYZ")

def _sum_digits(n: int) -> int:
    return n if n < 10 else n//10 + n%10

def cif_generate() -> str:
    first = random.choice(list(LETTER_GROUP | DIGIT_GROUP | ANY_GROUP))
    digits = [random.randint(0,9) for _ in range(7)]
    sum_even = digits[1] + digits[3] + digits[5]
    sum_odd = sum(_sum_digits(2*d) for d in (digits[0], digits[2], digits[4], digits[6]))
    total = sum_even + sum_odd
    cd_num = (10 - (total % 10)) % 10
    if first in LETTER_GROUP:
        control = CONTROL_LETTERS[cd_num]
    elif first in DIGIT_GROUP:
        control = str(cd_num)
    else:
        control = CONTROL_LETTERS[cd_num] if random.random() < 0.5 else str(cd_num)
    body = "".join(str(d) for d in digits)
    return f"{first}{body}{control}"

def get_cif(empresa: str) -> str:
    k = norm_txt(empresa)
    return map_empresa_to_cif.get(k) or cif_generate()



In [130]:
def nif_de_grupo(grupo_texto: str) -> str:
    return get_cif(grupo_texto)

In [131]:
def station_offset(id_estacion: str) -> float:
    rnd = random.Random(hash(id_estacion) & 0xffffffff)
    return float(rnd.uniform(STATION_OFFSET_MIN, STATION_OFFSET_MAX))

def band_by_power(potencia_max_kw: float | int | None) -> str:
    try:
        v = float(potencia_max_kw)
    except (TypeError, ValueError):
        v = np.nan
    return "DC" if pd.notna(v) and v > DC_THRESHOLD_KW else "AC"

def precio_base_mes(prov_norm: str, mes: int, band: str, id_estacion: str | None = None) -> float | None:
    val = None
    if id_estacion is not None and (id_estacion, mes, band) in precios_station_idx:
        val = precios_station_idx.get((id_estacion, mes, band))
    if (val is None or pd.isna(val)) and (prov_norm, mes, band) in precios_prov_idx:
        val = precios_prov_idx.get((prov_norm, mes, band))
    return float(val) if val is not None and not pd.isna(val) else None

def precio_diario(prov_norm: str, fecha: date, id_estacion: str, band: str) -> float:
    base = precio_base_mes(prov_norm, fecha.month, band, id_estacion)
    if base is None or pd.isna(base):
        candidatos = []
        if (prov_norm, fecha.month, band) in precios_prov_idx:
            v = precios_prov_idx.get((prov_norm, fecha.month, band))
            if v is not None and not pd.isna(v):
                candidatos.append(v)
        if not candidatos:
            candidatos = [v for (p, m, b), v in precios_prov_idx.items() if m == fecha.month and b == band and v is not None and not pd.isna(v)]
        base = float(np.mean(candidatos)) if candidatos else 0.45
    off = station_offset(id_estacion)
    noise = random.uniform(DAILY_NOISE_MIN, DAILY_NOISE_MAX)
    return round3(max(0.15, base + off + noise))

In [132]:
@dataclass
class Empresa:
    id: str
    nombre: str

@dataclass
class Usuario:
    id: str
    empresa_id: str
    nombre: str

empresas = [Empresa(id=f"EMP{i+1:03d}", nombre=f"Transporte_{i+1:02d} S.L.") for i in range(N_EMPRESAS_TRANSP)]
usuarios = []
for e in empresas:
    for j in range(USUARIOS_POR_EMPRESA):
        usuarios.append(Usuario(id=f"{e.id}-U{j+1:03d}", empresa_id=e.id, nombre=f"Usuario_{j+1:02d}_{e.id}"))

In [133]:
EST_POOL = est_std.reset_index(drop=True).copy()
EST_POOL["id_estacion"] = EST_POOL["id_estacion"].astype(str)

EST_POOL["lat"] = pd.to_numeric(EST_POOL["lat"].astype(str).str.replace(",", ".", regex=False), errors="coerce")
EST_POOL["lon"] = pd.to_numeric(EST_POOL["lon"].astype(str).str.replace(",", ".", regex=False), errors="coerce")

mask_coords = EST_POOL["lat"].between(-90, 90) & EST_POOL["lon"].between(-180, 180)
if mask_coords.any():
    EST_POOL = EST_POOL.loc[mask_coords].reset_index(drop=True)

EST_POOL["potencia_max_kw"] = pd.to_numeric(EST_POOL["potencia_max_kw"], errors="coerce")
EST_POOL["provincia_norm"] = EST_POOL["provincia"].map(prov_key)
EST_POOL["band"] = EST_POOL["potencia_max_kw"].map(band_by_power)

if len(EST_POOL) == 0:
    raise ValueError("EST_POOL vacío: revisa columnas de lat/lon en PuntosCarga.csv o el mapeo de columnas.")


In [134]:
IVA_TIPO = 0.21
IEE_TIPO = 0.0511

def calcular_importes(kwh: float, precio_unit: float) -> dict:
    kwh = max(0.0, float(kwh))
    precio_unit = round3(float(precio_unit))
    total = round2(kwh * precio_unit)
    mult = (1 + IEE_TIPO) * (1 + IVA_TIPO)
    base = round2(total / mult)
    iee  = round2(base * IEE_TIPO)
    iva  = round2((base + iee) * IVA_TIPO)
    ajuste = round2(total - (base + iee + iva))
    if ajuste != 0:
        iva = round2(iva + ajuste)
    return {"precio_unitario": precio_unit, "importe_total": total, "base_imponible": base, "iee": iee, "iva": iva}


In [135]:
def _to_float_locale(val):
    if pd.isna(val):
        return np.nan
    s = str(val).strip().replace(",", ".")
    try:
        return float(s)
    except Exception:
        return np.nan

def _coords_from_row(row):
    lat = _to_float_locale(row.get("lat", np.nan))
    lon = _to_float_locale(row.get("lon", np.nan))

    def _ok(la, lo):
        return (27.0 <= la <= 44.5) and (-20.0 <= lo <= 5.5)

    if not _ok(lat, lon) and _ok(lon, lat):
        lat, lon = lon, lat

    if not _ok(lat, lon):
        return np.nan, np.nan
    return lat, lon

def generar_ticket(empresa: Empresa, usuario: Usuario) -> dict:
    f = random_fecha(FECHA_INI, FECHA_FIN)
    h = random_hora()

    lat, lon = np.nan, np.nan
    attempts = 0
    row = None
    while attempts < 5 and (pd.isna(lat) or pd.isna(lon)):
        row = EST_POOL.sample(1).iloc[0]
        lat, lon = _coords_from_row(row)
        attempts += 1

    prov_norm = row["provincia_norm"]
    sid = str(row["id_estacion"])
    band = band_by_power(row.get("potencia_max_kw"))
    punit = precio_diario(prov_norm, f, sid, band)

    kwh = random_kwh()
    metodo = elegir_metodo_pago()
    imp = calcular_importes(kwh, punit)

    nif_operador = get_cif(row.get("empresa", ""))

    ticket = {
        "idTicket": f"T-{uuid.uuid4().hex[:12].upper()}",
        "idEmpresa": empresa.id,
        "empresaNombre": empresa.nombre,
        "idUsuario": usuario.id,
        "fechaEmision": str_fecha(f),
        "horaEmision": str_hora(h),
        "metodoPago": metodo,
        "estacion": {
            "id": sid,
            "provincia": row.get("provincia", ""),
            "municipio": row.get("municipio", ""),
            "direccion": row.get("direccion", ""),
            "lat": None if pd.isna(lat) else float(lat),
            "lon": None if pd.isna(lon) else float(lon),
            "empresa": row.get("empresa", ""),
            "nifEmpresa": nif_operador,
            "potenciaMaxKW": None if pd.isna(row.get("potencia_max_kw")) else float(row.get("potencia_max_kw")),
            "tarifa": band
        },
        "lineas": [
            {
                "producto": PRODUCTO,
                "kwh": kwh,
                "precioUnitario": imp["precio_unitario"],
                "importe": imp["importe_total"]
            }
        ],
        "baseImponible": imp["base_imponible"],
        "iee": imp.get("iee", 0.0),
        "iva": imp["iva"],
        "total": imp["importe_total"],
        "moneda": "EUR",
        "tipoDocumento": "Factura simplificada"
    }
    return ticket


In [136]:
def generar_todos() -> list[dict]:
    tickets = []
    for emp in empresas:
        us_emp = [u for u in usuarios if u.empresa_id == emp.id]
        for u in us_emp:
            for _ in range(TICKETS_POR_USUARIO):
                ok = False
                for _retry in range(8):
                    try:
                        t = generar_ticket(emp, u)
                        if isinstance(t, dict):
                            tickets.append(t)
                            ok = True
                            break
                    except Exception:
                        continue
                if not ok:
                    continue
    random.shuffle(tickets)
    return tickets

tickets = generar_todos()

In [137]:
with open(OUT_JSONL, "w", encoding="utf-8") as f:
    for tk in tickets:
        f.write(json.dumps(tk, ensure_ascii=False) + "\n")

with open(OUT_JSON, "w", encoding="utf-8") as f:
    json.dump(tickets, f, ensure_ascii=False, indent=2)

OUT_JSONL, OUT_JSON

(WindowsPath('data/tickets_ev_sinteticos.jsonl'),
 WindowsPath('data/tickets_ev_sinteticos.json'))

In [138]:
df_chk = pd.DataFrame([{
    "empresa":  t.get("idEmpresa"),
    "usuario":  t.get("idUsuario"),
    "fecha":    t.get("fechaEmision"),
    "hora":     t.get("horaEmision"),
    "metodo":   t.get("metodoPago"),

    "producto": t["lineas"][0].get("producto") if t.get("lineas") else None,
    "precio":   t["lineas"][0].get("precioUnitario") if t.get("lineas") else None,
    "litros":   t["lineas"][0].get("litros") if t.get("lineas") else None,
    "total":    t.get("total"),

    "est_id":        t.get("estacion", {}).get("id"),
    "est_nombre":    t.get("estacion", {}).get("nombre"),
    "est_grupo":     t.get("estacion", {}).get("grupo"),
    "est_nif":       t.get("estacion", {}).get("nifEmpresa"),
    "provincia":     t.get("estacion", {}).get("provincia"),
    "municipio":     t.get("estacion", {}).get("municipio"),
    "direccion":     t.get("estacion", {}).get("direccion"),
    "lat":           t.get("estacion", {}).get("lat"),
    "lon":           t.get("estacion", {}).get("lon"),
} for t in tickets])

for c in ["precio","litros","total","lat","lon"]:
    df_chk[c] = pd.to_numeric(df_chk[c], errors="coerce")

print("Tickets totales:", len(df_chk))
print(df_chk.head(5)) 

print("\nPor empresa:")
print(df_chk.groupby("empresa").size())

print("\nPor usuario:")
print(df_chk.groupby("usuario").size().head(10))

print("\nPor producto:")
print(df_chk.groupby("producto").agg(n=("producto","size"), p_med=("precio","mean")).reset_index())

print("\nPor grupo (marca):")
print(
    df_chk.groupby("est_grupo")
          .agg(tickets=("empresa","size"),
               p_med=("precio","mean"),
               litros=("litros","sum"),
               gasto=("total","sum"))
          .sort_values("gasto", ascending=False)
          .head(10)
)

print("\nTop 10 estaciones por gasto total:")
print(
    df_chk.groupby(["est_id","est_nombre","est_grupo"])
          .agg(tickets=("empresa","size"), gasto=("total","sum"))
          .sort_values("gasto", ascending=False)
          .head(10)
)

Tickets totales: 450
  empresa      usuario       fecha      hora              metodo  \
0  EMP003  EMP003-U002  2025-02-05  04:35:30  Tarjeta de empresa   
1  EMP001  EMP001-U001  2025-04-13  04:23:15  Tarjeta de empresa   
2  EMP002  EMP002-U003  2025-04-10  07:26:35  Tarjeta de empresa   
3  EMP002  EMP002-U001  2025-06-25  15:05:53  Tarjeta de empresa   
4  EMP002  EMP002-U003  2025-01-21  01:39:52  Tarjeta de empresa   

       producto  precio  litros  total                   est_id est_nombre  \
0  Electricidad   0.447     NaN  17.77  ES*ESX*E123070540300071       None   
1  Electricidad   0.449     NaN  34.54  ES*WEN*ESGASVALDEMORO14       None   
2  Electricidad   0.461     NaN  33.18   ES*SHE*E12128025310002       None   
3  Electricidad   0.440     NaN  18.29    ES*PRR*EES70900001501       None   
4  Electricidad   0.460     NaN  21.14            ES*REP*E13045       None   

  est_grupo    est_nif                    provincia               municipio  \
0      None  B09732520